<a href="https://colab.research.google.com/github/AI4ChemS/CHE1147/blob/main/tutorials/tutorial_05_classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Building Classification Models

## Machine learning for discovery of MOFs for Carbon Capture

In this tutorial, we will build simple machine learning classification models to identify promising metal–organic frameworks (MOFs) for carbon capture applications.

Carbon capture is a complex challenge involving multiple material considerations — including pore geometry, surface chemistry, mechanical and thermal stability, water stability, and economic feasibility.

To make the problem tractable, we will focus on two experimentally measurable properties that strongly influence performance:
1) CO₂ uptake at low pressure, which reflects adsorption capacity and selectivity.

2) Water stability, which determines a material’s long-term durability and reusability.

By the end of this tutorial, you’ll have trained and evaluated models that can distinguish between promising and non-promising MOFs based on these properties — demonstrating how data-driven methods can accelerate materials discovery.

# 0. Setup programming environment

### 0.1 Installing packages

If you are running this notebook locally after cloning [UofT-CHE1147](https://github.com/AI4ChemS/CHE1147) repo, simply install the conda environment

```python
conda env create -f environment.yml
conda activate che1147


If you are running this notebook on Google Colab, please uncomment the lines below (remove the `#`) and execute the cell.

In [ ]:
import os, sys, urllib.request

!pip install -r https://github.com/AI4ChemS/CHE1147/raw/refs/heads/main/requirements_colab.txt

CHE1147_DIR = "/content/che1147_files"
os.makedirs(CHE1147_DIR, exist_ok=True)

data_URL_1 = "https://github.com/AI4ChemS/CHE1147/raw/refs/heads/main/data/MOF_CoRE2019.csv"
# Corrected data_URL_2 to use the raw GitHub link
data_URL_2 = "https://github.com/AI4ChemS/CHE1147/raw/refs/heads/main/data/all_data_ws.csv"
descriptors_URL = "https://github.com/AI4ChemS/CHE1147/raw/refs/heads/main/tutorials/MOF_descriptors.py"

local_descriptor_path = os.path.join(CHE1147_DIR, "MOF_descriptors.py")
urllib.request.urlretrieve(descriptors_URL, local_descriptor_path)

local_data_path_1 = os.path.join(CHE1147_DIR, "MOF_CoRE2019.csv")
urllib.request.urlretrieve(data_URL_1, local_data_path_1)

local_data_path_2 = os.path.join(CHE1147_DIR, "water_stability_chemunity_v0.1.0.csv")
# Use the corrected data_URL_2 for downloading the water stability data
urllib.request.urlretrieve(data_URL_2, local_data_path_2)

if CHE1147_DIR not in sys.path:
    sys.path.append(CHE1147_DIR)

### 0.2 Import packages we will need

> Note, if you are using colab, you may need to install some of the packages.

In [ ]:
# basics
import os
import numpy as np
import pprint as pp

# pandas is used to read/process data
import pandas as pd
from ydata_profiling import ProfileReport

# machine learning dependencies
# scaling of data
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler
# train/test split
from sklearn.model_selection import train_test_split
# model selection
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
# the KRR model
from sklearn.kernel_ridge import KernelRidge
# linear model
from sklearn.linear_model import LinearRegression, SGDRegressor
# pipeline to streamline modeling pipelines
from sklearn.pipeline import Pipeline
# principal component analysis
from sklearn.decomposition import PCA
# polynomial kernel
from sklearn.metrics.pairwise import polynomial_kernel
# Dummy model as baseline
from sklearn.dummy import DummyClassifier, DummyRegressor
# Variance Threshold for feature selection
from sklearn.feature_selection import VarianceThreshold, SelectFromModel
# metrics to measure model performance
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             mean_absolute_error, mean_squared_error, max_error, mean_absolute_percentage_error)
# confusion matrix
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, ConfusionMatrixDisplay
# xg boost classifer
from xgboost import XGBClassifier



# save/load models
import joblib

# For the permutation importance implementation
from joblib import Parallel
from joblib import delayed
from sklearn.metrics import check_scoring
from sklearn.utils import Bunch
from sklearn.utils import check_random_state
from sklearn.utils import check_array

# plotting
import seaborn as sns
import matplotlib.pyplot as plt
%matplotlib inline

### 0.3 Fix the random seed

In [ ]:
# add code here
RANDOM_SEED = 1
np.random.seed(RANDOM_SEED)

### 0.4 Import the data

We will be working with two datasets:

1. We will use the dataset from the previous tutorial, originating from the publication ["Understanding the diversity of the metal-organic framework     ecosystem"](https://doi.org/10.1038/s41467-020-17755-8) to featurize our MOFs (using geometric descriptors and RACs). This dataset also contains simulated CO2 uptake values, which will be the basis for our target in Task #1

2. Our second task will be to predict MOF water stability. Water stability cannot be effectively simulated, and therefore the labels we use are experimental results which have been extracted from literature. They originate from the paper ["MOF-ChemUnity: Unifying metal-organic framework data using large language models"](https://chemrxiv.org/engage/chemrxiv/article-details/6838df8bc1cb1ecda036f363).

In [ ]:
# load data locally

running_in_colab = 'google.colab' in str(get_ipython())

if running_in_colab:
    DATA_DIR = "/content/che1147_files"
    CO2_FILE = os.path.join(DATA_DIR, "MOF_CoRE2019.csv")
    # Correct the water stability file name and URL to the raw version
    WS_FILE  = os.path.join(DATA_DIR, "water_stability_chemunity_v0.1.0.csv") # Use the local name
else:
    DATA_DIR = "../data"
    CO2_FILE = os.path.join(DATA_DIR, "MOF_CoRE2019.csv")
    WS_FILE  = os.path.join(DATA_DIR, "water_stability_chemunity_v0.1.0.csv") # Use the local name


try:
    df_co2 = pd.read_csv(CO2_FILE)
    df_ws  = pd.read_csv(WS_FILE)
    print(f"✅ Loaded CO₂ dataset from {CO2_FILE}, shape = {df_co2.shape}")
    print(f"✅ Loaded water stability dataset from {WS_FILE}, shape = {df_ws.shape}")

except FileNotFoundError:
    # Fallback: try to load from GitHub raw URLs
    try:
        df_co2 = pd.read_csv("https://github.com/AI4ChemS/CHE1147/raw/refs/heads/main/data/MOF_CoRE2019.csv")
        # Correct the water stability URL to the raw version
        df_ws  = pd.read_csv("https://github.com/AI4ChemS/CHE1147/raw/refs/heads/main/data/all_data_ws.csv")
        print("🌐 Loaded both datasets from GitHub.")
        print(f"CO₂ shape = {df_co2.shape}, Water stability shape = {df_ws.shape}")
    except Exception as e:
        print("❌ Could not find datasets locally or on GitHub.", e)

Let's take a look!

In [ ]:
df_co2.head()

In [ ]:
df_ws.head()

We will now set our water stability dataset aside and only work with the CO2 dataset (we'll come back to it during Task #2).

# Task 1: Identifying MOFs with Promising CO2 Uptakes

## 1.1: Data Cleaning and Prep

### 1.1.1 Basic Cleaning

In [ ]:
# Drop duplicate rows
#FILL ME

# Drop rows with any NaN values
#FILL ME

### 1.1.2: Splitting our data into classes and loading features/target

First, since this is a tutorial on classification, lets create two classes from our data:

1) Promising - a MOF with CO2 Uptake at LP > 2mmol/g
2) Not Promising - a MOF with CO2 Uptake at LP < 2mmol/g

We choose 2mmol/g as our criteria for a "promising" MOF for CO2 capture. This is based on [Mahajan et al. (2022)](https://www.sciencedirect.com/science/article/pii/S2213343722018036), who stated that for a promising candidate for
carbon capture, anything greater than 2 mmol CO2/g adsorbent is acceptable.

In [ ]:
df_co2['Promise'] = df_co2['CO2 uptake at 0.15 bar and 298K'].apply(lambda x: 'Promising' if x > 2 else 'Not Promising')
df_co2['Promise'].value_counts()

Note, we have a class imbalance here: about 65% are Not Promising

Let's also define our features and our targets now as global variables:

In [ ]:
# name of descriptors
from MOF_descriptors import geometric_descriptors, linker_descriptors, metalcenter_descriptors, functionalgroup_descriptors, summed_linker_descriptors, summed_metalcenter_descriptors, summed_functionalgroup_descriptors

TARGET = "Promise"
FEATURES = (
    geometric_descriptors
    + summed_functionalgroup_descriptors
    + summed_linker_descriptors
    + summed_metalcenter_descriptors
)

### 1.1.3 Train/Test Split

We know we have a class imbalance, so as demonstrated in Tutorial 1, we will use a stratified split to seperate our train/test data.

In [ ]:
from sklearn.model_selection import train_test_split

X = df_co2[FEATURES]
y = df_co2[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_SEED, stratify=y
)


# Plot the class distrubition:
fig, axes = plt.subplots(1, 2, figsize=(12, 6))

sns.countplot(x=y_train, ax=axes[0])
axes[0].set_title('Class Distribution in Training Set')
axes[0].set_xlabel('Promise')
axes[0].set_ylabel('Count')

sns.countplot(x=y_test, ax=axes[1])
axes[1].set_title('Class Distribution in Testing Set')
axes[1].set_xlabel('Promise')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.show()

 $\color{Aqua}{\textsf{Short question}}$
- Why does keeping the class distributions the same between training and testing improve model performance?
<details>
<summary> <font color='green'>Click here for a hint</font></summary>
<ul>
    <li>Learns from the same proportions of each class it will later see in evaluation</li>
    <li>Avoids being biased toward the majority class during training. </li>
</ul>
</details>

### 1.1.4 Standardize Data
(Always after train-test split, or else its data leakage!)

In [ ]:
# Initialize the StandardScaler
scaler = StandardScaler()

# Fit the scaler on the training data and transform both training and testing data
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

### 1.1.5 Make Labels Numeric

In [ ]:
# Convert target labels to numerical values (0 or 1)
y_train_numeric = y_train.apply(lambda x: 1 if x == 'Promising' else 0)
y_test_numeric = y_test.apply(lambda x: 1 if x == 'Promising' else 0)

## 1.2: EDA and Feature Selection

### 1.2.1: Remove Redundant Columns

In [ ]:
# Initialize VarianceThreshold with a threshold of 0
selector = VarianceThreshold(threshold=0)

# Fit on the training data and transform both training and testing data
X_train_reduced = selector.fit_transform(X_train)
X_test_reduced = selector.transform(X_test)

# Get the names of the selected features
selected_features = X_train.columns[selector.get_support()]

# Update the feature list and DataFrames
FEATURES = selected_features.tolist()
X_train = pd.DataFrame(X_train_reduced, columns=FEATURES, index=X_train.index)
X_test = pd.DataFrame(X_test_reduced, columns=FEATURES, index=X_test.index)

print(f"Original number of features: {X.shape[1]}")
print(f"Number of features after removing zero variance features: {X_train.shape[1]}")
print("\nUpdated feature list:")
print(FEATURES)

### 1.2.2: Remove Highly Correlated Features

In [ ]:
# Calculate the correlation matrix
corr_matrix = X_train.corr().abs()

# Select upper triangle of correlation matrix
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))

# Find features with correlation greater than 0.90
to_drop = [column for column in upper.columns if any(upper[column] > 0.90)]

# Drop features
X_train = X_train.drop(columns=to_drop)
X_test = X_test.drop(columns=to_drop)

print(f"Original number of features: {len(FEATURES)}")
print(f"Number of features after removing highly correlated features: {X_train.shape[1]}")
print("\nDropped features:")
print(to_drop)

# Update the global FEATURES list
FEATURES = X_train.columns.tolist()

$\color{Aqua}{\textsf{Short question}}$
- Why do we remove features that are correlated with eachother?
- Why are we looking at correlations between features? Shouldn't we be interested in how the features are correlated to the targets?

<details>
<summary> <font color='green'>Click here for a hint</font></summary>
<ul>
    <li> Because highly correlated features are redundant. They provide overlapping information, and only one of these overlapping features is needed to give the model this signal. Redundant features can make the model less stable, inflate coefficient magnitudes (especially in linear models), and reduce interpretability without improving performance.</li>
    <li> We do also care about feature–target correlation for predictive power, but we saw this last week. An important step in feature engineering is checking feature–feature correlation, which ensures the model learns independent, non-redundant signals.
    <li> Feature–target correlation makes less sense for classification, since the target is categorical — the correlation only captures mean differences between classes, not the true strength or shape of the relationship.
</ul>
</details>

## 1.3 Classification Metrics

Let's write the function `get_classification_metrics(model, X, y_true)` that compute the metrics and return this dictionary for a given model. Include these metrics in your function:

$$
\mathrm{Accuracy} = \frac{TP + TN}{TP + TN + FP + FN} \\
\mathrm{Precision} = \frac{TP}{TP + FP} \\
\mathrm{Recall} = \frac{TP}{TP + FN} \\
\mathrm{F1\text{-}Score} = 2 \times \frac{\mathrm{Precision} \times \mathrm{Recall}}{\mathrm{Precision} + \mathrm{Recall}}
$$


In [ ]:
def get_classification_metrics(model, X, y_true):
    """
    Get a dictionary with basic classification metrics:

    model: sklearn-like model with predict method
    X: feature matrix
    y_true: ground truth binary labels ('Promising' or 'Not Promising')
    """
    # Get model predictions
    # Convert predictions to the same format as y_true
    y_pred_numeric = model.predict(X)
    y_pred = pd.Series(y_pred_numeric).apply(lambda x: 'Promising' if x > 0.5 else 'Not Promising')
    y_pred.index = y_true.index # Ensure indices align

    # Compute confusion matrix components using element-wise comparison
    TP = ((y_true == 'Promising') & (y_pred == 'Promising')).sum()
    FP = # FILL ME
    FN = # FILL ME
    TN = # FILL ME


    # Compute metrics manually
    accuracy = # FILL ME
    precision = # FILL ME
    recall = # FILL ME
    f1 = # FILL ME

    metrics_dict = {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1
    }

    return metrics_dict

$\color{Aqua}{\textsf{Short question}}$
- What do each of these metrics mean?

<details>
<summary> <font color='green'>Click here for a hint</font></summary>
<ul>
    <li> Accuracy measures overall correctness.
    <li> Precision measures how reliable the positive predictions are.
    <li> Recall measures how many actual positives were captured.
    <li>F1-score balances precision and recall — especially useful when the classes are imbalanced.

</ul>
</details>

## 1.4: Logistic Regression

Classification isn’t fundamentally different from regression — it’s just regression with a different output interpretation and loss function. To demonstrate, you will fill in the sigmoid func
tion and the log-loss function for this custom logistic regression class.

### 1.4.1: Logistic Regression From Scratch

### The sigmoid converts any real value into a probability between 0 and 1:
$$
\sigma(z) = \frac{1}{1 + e^{-z}} \\
$$

It maps large negative values → 0, large positive values → 1.



### The log-loss (binary cross-entropy) measures how close predicted probabilities are to the true labels:
$$
\mathrm{Binary\ Cross\text{-}Entropy\ Loss} = -\frac{1}{N} \sum_{i=1}^{N} \left[ y_i \log(\hat{y}_i) + (1 - y_i) \log(1 - \hat{y}_i) \right]
$$
It heavily penalizes confident wrong predictions


In [ ]:
import numpy as np

class MyLogisticRegression:
    def __init__(self, lr=1e-1, n_iters=2000, tol=1e-8, random_state=0):
        """
        Simple Logistic Regression trained using Gradient Descent on Log-Loss.

        Parameters
        ----------
        lr : float
            Learning rate for gradient descent.
        n_iters : int
            Maximum number of iterations for gradient descent.
        tol : float
            Stopping criterion based on loss improvement.
        random_state : int
            Seed for reproducibility.
        """
        self.lr = lr
        self.n_iters = n_iters
        self.tol = tol
        self.random_state = random_state

        self.coef_ = None
        self.intercept_ = 0.0

    # ----------------------------------------------------------------------
    # STEP 1: Sigmoid function
    # ----------------------------------------------------------------------
    @staticmethod
    def _sigmoid(z):
        """
        TODO (students): Implement the sigmoid function.
        """
        return # FILL ME
        raise NotImplementedError("_sigmoid not implemented yet.")

    # ----------------------------------------------------------------------
    # STEP 2: Log-loss function
    # ----------------------------------------------------------------------
    @staticmethod
    def _log_loss(p, y):
        """
        TODO (students): Implement the binary cross-entropy loss.
        L = - mean( y*log(p) + (1-y)*log(1-p) )
        """
        eps = 1e-12
        p = #FILL ME
        L = #FILL ME
        return L
        raise NotImplementedError("_log_loss not implemented yet.")

    # ----------------------------------------------------------------------
    # STEP 3: Training (Gradient Descent)
    # ----------------------------------------------------------------------
    def fit(self, X, y):
        """
        Fit the logistic regression model on (X, y) using gradient descent.
        """
        rng = np.random.default_rng(self.random_state)
        X = np.asarray(X, dtype=float)
        y = np.asarray(y, dtype=float).reshape(-1,)

        # Add bias (intercept) column
        Xp = np.c_[np.ones((X.shape[0], 1)), X]

        # GRADIENT DESCENT
        n_features = Xp.shape[1]
        theta = rng.normal(scale=0.01, size=n_features)
        prev_loss = np.inf
        for i in range(self.n_iters):
            # --- Forward pass ---
            z = Xp @ theta
            p = self._sigmoid(z)

            # --- Compute residuals and loss ---
            residuals = p - y
            loss = self._log_loss(p, y)

            # --- Convergence check ---
            if abs(prev_loss - loss) < self.tol:
                break
            prev_loss = loss

            # --- Compute gradient ---
            grad = (Xp.T @ residuals) / len(y)
            theta -= self.lr * grad

        # Unpack learned parameters
        self.intercept_ = float(theta[0])
        self.coef_ = theta[1:]
        return self


    # ----------------------------------------------------------------------
    # STEP 4: Predictions
    # ----------------------------------------------------------------------
    def predict_proba(self, X):
        """
        Return predicted probabilities for class 0 and class 1.
        """
        X = np.asarray(X, dtype=float)
        X = np.c_[np.ones((X.shape[0], 1)), X]  # always include bias column
        z = X @ np.r_[self.intercept_, self.coef_]
        p1 = self._sigmoid(z)
        return np.c_[1 - p1, p1]

    def predict(self, X, threshold=0.5):
        """
        Predict binary class labels (0 or 1).
        """
        return (self.predict_proba(X)[:, 1] >= threshold).astype(int)

$\color{Aqua}{\textsf{Short question}}$
- In our linear regression class, we had two ways to optimize the model parameters - "normal" and "gradient descent". Why do we only use one here?

<details>
<summary> <font color='green'>Click here for a hint</font></summary>
<ul>
    <li> Unlike linear regression, logistic regression has no closed form solution! Logistic regression’s loss involves a sigmoid, making it nonlinear — so we can’t solve it exactly, and we must use iterative methods like gradient descent.

</ul>
</details>

Now, train your linear regressor on your classification task

In [ ]:
# Instantiate and train the MyLogisticRegression model
logistic_reg_model = MyLogisticRegression(lr=1e-1, n_iters=2000, random_state=RANDOM_SEED)
logistic_reg_model.fit(X_train_scaled, y_train_numeric)

# Evaluate the model using the classification metrics function
# The get_classification_metrics function expects the model, X (scaled features), and y_true (string labels)
metrics = get_classification_metrics(logistic_reg_model, X_test_scaled, y_test)

print("Classification Metrics for MyLogisticRegression (from scratch):")
for metric, value in metrics.items():
    print(f"{metric}: {value:.4f}")

Generate a confusion matric using sklearn

In [ ]:
# Get predictions
y_pred_numeric = logistic_reg_model.predict(X_test_scaled)

# Calculate the confusion matrix
cm = confusion_matrix(y_test_numeric, y_pred_numeric)

# Get the class labels
class_labels = ['Not Promising', 'Promising']

# Display the confusion matrix
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_labels)

fig, ax = plt.subplots(figsize=(8, 6))
disp.plot(cmap=plt.cm.Blues, ax=ax)
plt.title('Confusion Matrix for MyLogisticRegression')
plt.show()

$\color{Aqua}{\textsf{Short question}}$
- Based on what we are searching for (Promising MOFs for carbon capture) which of these four categories do we want to minimize the most? How could change our model to do this?

<details>
<summary> <font color='green'>Click here for a hint</font></summary>
<ul>
    <li> We want to minimize the number of promising MOFs that are predicted to be not promising! (Bottom left corner of confusion matrix, false negatives). We don't want to let any slip by.
    <li> We could lower the decision boundary to predict “positive” more often → fewer false negatives (at the cost of more false positives). This could reduce overall model performance, but w.r.t. what's important to us, it makes sense

</ul>
</details>

In [ ]:
from sklearn.metrics import roc_curve, auc
import matplotlib.pyplot as plt

# Get the predicted probabilities for the positive class ('Promising' which is 1)
# The predict_proba method of MyLogisticRegression returns probabilities for class 0 and 1
y_pred_proba = logistic_reg_model.predict_proba(X_test_scaled)[:, 1]

# The roc_curve function expects binary true labels (0 or 1)
y_test_numeric = y_test.apply(lambda x: 1 if x == 'Promising' else 0)

# Calculate ROC curve points
fpr, tpr, thresholds = roc_curve(y_test_numeric, y_pred_proba)

# Calculate AUC
roc_auc = auc(fpr, tpr)

# Plot the ROC curve
plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {roc_auc:.2f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Random Guess')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC) Curve')
plt.legend(loc='lower right')
plt.show()

print(f"AUC: {roc_auc:.4f}")

# Task 2: Identifying Water Stable MOFs

Now that we've built a model for predicting MOFs with promising CO2 uptakes, let's move on to predicting MOF water stability! Water stability more difficult to predict, and we also have less data points, so we will use a more complicated classification model - XGBoost.

## 2.1: Data Prep and Splitting

Note: We will not be doing feature selection on this dataset. This could be something you can explore on your own! However, we will remove two features we know are redundant:

density_2 (g/cm^3) - This is a duplicate feature.

V_cell_2 (A^3) - Cell volume depends on how the unit cell is defined, not the material itself. It is not a feature with meaningful signal.


### 2.1.1: Set features and targets

In [ ]:
# Define the target for water stability
TARGET_WS = 'label' # Assuming 'label' column contains the water stability labels (0 or 1)

# Define features for water stability (all columns except target and ID)
# Get all columns from df_ws
all_ws_cols = df_ws.columns.tolist()

# Remove the target and ID columns from the feature list
FEATURES_WS = [col for col in all_ws_cols if col not in [TARGET_WS, 'ID']]

# --- Remove some specific features ---
features_to_drop = ['V_cell_2 (A^3)', 'density_2 (g/cm^3)']
FEATURES_WS = [col for col in FEATURES_WS if col not in features_to_drop]

# Create the feature matrix (X_ws) and target vector (y_ws)
X_ws = df_ws[FEATURES_WS]
y_ws = df_ws[TARGET_WS]

print(f"Shape of features (X_ws) for water stability task: {X_ws.shape}")
print(f"Shape of target (y_ws) for water stability task: {y_ws.shape}")

# Display the distribution of the new target variable
print("\nWater Stability Class Distribution:")
print(y_ws.value_counts())

We have a pretty large class imbalance for this dataset!

### 2.2.2: Split data into train and test

In [ ]:
# Stratified train-test split for water stability data
X_ws_train, X_ws_test, y_ws_train, y_ws_test = train_test_split(
    X_ws, y_ws, test_size=0.2, random_state=1, stratify=y_ws
)

## 2.2: Baseline Model

Here, we will use a dummy model again as our baseline:

$\color{Aqua}{\textsf{Short question}}$
- Does this confusion matrix reflect what we know about the class imbalance in our dataset?
- What will the accuracy of our dummy model be with this dataset split?

<details>
<summary> <font color='green'>Click here for a hint</font></summary>
<ul>
    <li> Yes! We see in the confusion matrix, the model predicts “Stable” much more often (245 vs 77)
    <li> Since our class split is ~2:1 in favour of the positive class, a dummy model that predicts "Stable" for everything would score an accuracy of 0.67. We see our model is not much better. We need to address this going forward

</ul>
</details>

In [ ]:
# Instantiate DummyClassifier (strategy='most_frequent' is a common baseline for imbalanced data)
dummy_model = DummyClassifier(strategy="most_frequent", random_state=RANDOM_SEED)

# Train the dummy model on the training data
dummy_model.fit(X_ws_train, y_ws_train)

# Make predictions on the test set
y_ws_pred_dummy = dummy_model.predict(X_ws_test)

# Evaluate using sklearn's metrics
accuracy_dummy = accuracy_score(y_ws_test, y_ws_pred_dummy)
precision_dummy = precision_score(y_ws_test, y_ws_pred_dummy, zero_division=0) # Add zero_division=0 to handle cases with no positive predictions
recall_dummy = recall_score(y_ws_test, y_ws_pred_dummy, zero_division=0)
f1_dummy = f1_score(y_ws_test, y_ws_pred_dummy, zero_division=0)


print("\nClassification Metrics for Dummy Baseline Model:")
print(f"Accuracy: {accuracy_dummy:.4f}")
print(f"Precision: {precision_dummy:.4f}")
print(f"Recall: {recall_dummy:.4f}")
print(f"F1-Score: {f1_dummy:.4f}")

# Display Confusion Matrix
cm_dummy = confusion_matrix(y_ws_test, y_ws_pred_dummy)

class_labels_ws = ['Unstable (0)', 'Stable (1)'] # Define if not found


disp_dummy = ConfusionMatrixDisplay(confusion_matrix=cm_dummy, display_labels=class_labels_ws)

fig_dummy, ax_dummy = plt.subplots(figsize=(8, 6))
disp_dummy.plot(cmap=plt.cm.Blues, ax=ax_dummy)
plt.title('Confusion Matrix for Dummy Baseline Model')
plt.show()

## 2.3 Set up Pipeline

Create a pipeline that scales your features and uses XGBoost classifer with default parameters.

In [ ]:
# Define the pipeline
pipeline = # FILL ME

# Train the XGBoost model using the pipeline on the training data
# Using X_ws_train and y_ws_train which contain all original features for WS task before manual feature selection steps
pipeline.fit(X_ws_train, y_ws_train)


In [ ]:
# Get the predicted probabilities for the positive class (class 1) from the pipeline
y_ws_pred_proba_pipeline = pipeline.predict_proba(X_ws_test)[:, 1]

# Calculate ROC curve points
fpr_pipeline, tpr_pipeline, thresholds_pipeline = roc_curve(y_ws_test, y_ws_pred_proba_pipeline)

# Calculate AUC
roc_auc_pipeline = auc(fpr_pipeline, tpr_pipeline)

# Plot the ROC curve
plt.figure(figsize=(8, 6))
plt.plot(fpr_pipeline, tpr_pipeline, color='darkorange', lw=2, label=f'ROC curve (AUC = {roc_auc_pipeline:.2f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Random Guess')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC) Curve for XGBoost Pipeline')
plt.legend(loc='lower right')
plt.show()

print(f"AUC for XGBoost Pipeline: {roc_auc_pipeline:.4f}")